---
---
---

# Summary

---
---
---

<br>

> | **_Language:_** python@3.12.10 |
> | - |

<br>

> | **_Source:_** `notebook/summary.ipynb` |
> | - |

<br>

> | **_Configurations:_** `lib/config` |
> | - |

<br>

> | **_Libraries:_** `lib/utils` |
> | - |

<br>

## Dependencies

### Packages

In [ ]:
from IPython.display import Image, display
from google.cloud import bigquery, storage
from loguru import logger
from matplotlib.ticker import MaxNLocator, PercentFormatter
from paddleocr import TextRecognition
from pathlib import Path

import contextily as ctx
import cv2
import geopandas as gpd
import io
import joblib
import math
import matplotlib.pyplot as plt
import numpy as np
import paddle
import pandas as pd
import shutil
import sys
import warnings

ROOT_PATH = Path.cwd().resolve()
if ROOT_PATH.name in ["notebook"]:
    ROOT_PATH = ROOT_PATH.parent
if str(ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(ROOT_PATH))

from config import config as the_config
from utils import etl as the_etl
from utils import pipeline as the_pipeline

from jobs.data_extract import extract_data

# -------------------------

# Settings
%matplotlib inline
the_config.refresh_logging()
paddle.disable_signal_handler()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.3f}".format)
warnings.filterwarnings("ignore")

### Constants

In [ ]:
# GCP
BQ_CLIENT = bigquery.Client(project=the_config.GCP_PROJECT)
GCS_CLIENT = storage.Client(project=the_config.GCP_PROJECT)
GCS_BUCKET = GCS_CLIENT.bucket(the_config.GCS_BUCKET)

### Paths

In [ ]:
# Paths
for path in the_config.PATHS:
    the_config.ensure_path(path)

## Data Collection

### Context

In [ ]:
locations = []
for province, cities in the_config.LOCATIONS.items():
    for city_key, info in cities.items():
        locations.append({
            "name": info["city"],
            "latitude": info["latitude"],
            "longitude": info["longitude"],
        })

stations = []
for province, tags in the_config.STATIONS.items():
    for tag, info in tags.items():
        stations.append({
            "name": info["city"],
            "latitude": info["latitude"],
            "longitude": info["longitude"],
        })

locations_df = pd.DataFrame(locations)
g_locations_df = gpd.GeoDataFrame(
	locations_df,
	geometry=gpd.points_from_xy(
        locations_df.longitude,
        locations_df.latitude
    ),
	crs="EPSG:4326",
).to_crs(epsg=3857)

stations_df = pd.DataFrame(stations)
g_stations_df = gpd.GeoDataFrame(
	stations_df,
	geometry=gpd.points_from_xy(
        stations_df.longitude,
        stations_df.latitude
    ),
	crs="EPSG:4326",
).to_crs(epsg=3857)

# -------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 6))
g_locations_df.plot(
    ax=ax1,
    color="blue",
    markersize=60,
    edgecolor="black",
)

for _, row in g_locations_df.iterrows():
    ax1.annotate(
        row["name"],
        (row.geometry.x, row.geometry.y),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        bbox=dict(fc="white", alpha=0.7, ec="none", pad=0.2),
    )
ctx.add_basemap(ax1, source=ctx.providers.OpenTopoMap)
ax1.set_title("Reference Locations")
ax1.set_axis_off()

# -------------------------

g_stations_df.plot(
    ax=ax2,
    color="red",
    markersize=60,
    edgecolor="black",
)

for _, row in g_stations_df.iterrows():
    ax2.annotate(
        row["name"],
        (row.geometry.x, row.geometry.y),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        bbox=dict(fc="white", alpha=0.7, ec="none", pad=0.2),
    )
ctx.add_basemap(ax2, source=ctx.providers.OpenTopoMap)
ax2.set_title("Meteorological Stations")
ax2.set_axis_off()

plt.tight_layout()
plt.show()

### Extraction

In [ ]:
extract_data(n=1)

In [ ]:
files = list(the_config.TMP_IMG_PATH.iterdir())
filename = files[0]
display(Image(filename=str(filename)))

### Transformation

In [ ]:
# [!]
img_cv = cv2.imread(str(filename))
img_cv = cv2.resize(
    img_cv,
    None,
    fx=2,
    fy=2,
    interpolation=cv2.INTER_LINEAR
)

hsv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)
green_mask = cv2.inRange(
    hsv,
    np.array([35, 50, 50]),
    np.array([85, 255, 255]),
)
green = np.full_like(green_mask, 255)
green[green_mask > 0] = 0
blue_mask = cv2.inRange(
    hsv,
    np.array([90, 50, 50]),
    np.array([130, 255, 255]),
)
blue = np.full_like(blue_mask, 255)
blue[blue_mask > 0] = 0

In [ ]:
plt.figure(figsize=(16, 4))
plt.imshow(green, cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 4))
plt.imshow(blue, cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# [!]
green_rois = the_etl.extract_rois(img_cv, green_mask, "green")
blue_rois  = the_etl.extract_rois(img_cv, blue_mask, "blue")
all_rois = sorted(green_rois + blue_rois, key=lambda r: r["bbox"][0])

In [ ]:
debug = img_cv.copy()
green_idx = 1
blue_idx = 1

for roi in all_rois:

    x1, y1, x2, y2 = roi["bbox"]
    if roi["type"] == "green":
        box_color = (0, 255, 0)
        letter = "G"
        idx = green_idx
        green_idx += 1
    else:
        box_color = (255, 0, 0)
        letter = "B"
        idx = blue_idx
        blue_idx += 1

    cv2.rectangle(debug, (x1, y1), (x2, y2), box_color, 2)
    (letter_w, letter_h), _ = cv2.getTextSize(
		letter,
		cv2.FONT_HERSHEY_SIMPLEX,
		1.4,
		3
	)
    (label_w, label_h), baseline = cv2.getTextSize(
		f"{letter}{idx}",
		cv2.FONT_HERSHEY_SIMPLEX,
		1.4,
		3
	)

plt.figure(figsize=(16, 4))
plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# [!]
pieces = the_etl.get_pieces(img_cv, green_mask, all_rois)

In [ ]:
for i, piece in enumerate(pieces, start=1):

    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    axes[0].imshow(cv2.cvtColor(piece["image"], cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Image ({i})")
    axes[0].axis("off")
    axes[1].imshow(piece["green_mask"], cmap="gray")
    axes[1].set_title(f"Mask ({i})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# [!]
MODEL_OCR = TextRecognition(
    model_name=the_config.CV_OCR_MODEL,
    device=the_config.CV_DEVICE
)

In [ ]:
display(Image(filename))

# [!]
values = the_etl.ocr_predict(
    MODEL_OCR,
    the_config.CV_OCR_FIELDS,
    pieces,
    verbose=True
)

### Loading

In [ ]:
query = f"""
	SELECT * FROM `{the_config.BQ_TABLE}`
	ORDER BY TIMESTAMP(
		DATETIME(year, month, day, hour, minute, 0)
	)
"""
df = BQ_CLIENT.query(query).to_dataframe()
display(df.info())

### Benchmarking

In [ ]:
year_months = (
    df[["year", "month"]]
    .drop_duplicates()
    .sort_values(["year", "month"])
)
n_cols = 4
n_rows = math.ceil(len(year_months) / n_cols)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 4 * n_rows)
)

axes = axes.flatten()
for ax, (_, row) in zip(axes, year_months.iterrows()):

    year = row["year"]
    month = row["month"]
    counts = (
        df[(df["year"] == year) & (df["month"] == month)]
        .groupby("day")
        .size()
    )
    hours = counts / (the_config.ML_DAILY_COLL / 24)

    ax.bar(counts.index, hours.values)
    ax.set_title(f"{year}-{month}")
    ax.set_xticks(counts.index)
    ax.set_xlabel("Days")
    ax.set_ylabel("Hours")
    ax.xaxis.set_major_locator(
        MaxNLocator(nbins=15, integer=True)
	)

for ax in axes[len(year_months):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
conf_cols = [c for c in df.columns if c.startswith("conf_")]
plt.figure(figsize=(10, 6))
plt.plot(
    df[conf_cols],
    linewidth=0.5,
	alpha=0.5
)

plt.title("OCR Confidence")
plt.xlabel("Sample")
plt.ylabel("Confidence (%)")
plt.gca().yaxis.set_major_formatter(
    PercentFormatter(xmax=1.0, decimals=0)
)
plt.grid(True, axis="y", alpha=0.5)
plt.legend(conf_cols, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# [!]
shutil.rmtree(the_config.TMP_IMG_PATH)

## Inference

### DataFrame

In [ ]:
# [!]
# DataFrame
query = f"""
	SELECT * FROM `{the_config.BQ_TABLE}`
	ORDER BY TIMESTAMP(
		DATETIME(year, month, day, hour, minute, 0)
	) DESC
	LIMIT {the_config.INF_ROWS * 2}
"""
inf_df = BQ_CLIENT.query(query).to_dataframe()
display(inf_df.info())

### Feature Engineering

In [ ]:
# [!]
# Feature Engineering -- Classification
clf_df = inf_df.copy()
clf_df, target_clf = the_pipeline.feature_engineering_clf(
	clf_df, inference=True
)
clf_df = clf_df.assign(
	timestamp=lambda x: pd.to_datetime(
		x[["year", "month", "day", "hour", "minute"]]
	)
)

# [!]
# Feature Selection -- Classification
to_drop = [
	*the_config.CLASSIFICATION["to_drop"],
	"timestamp"
]
to_drop += [c for c in clf_df.columns if c.lower().startswith("conf_")]
X_clf = clf_df.drop(columns=to_drop)

logger.debug(f"[LIST] Targets ({len(target_clf)}): {target_clf}")

In [ ]:
# [!]
# Feature Engineering -- Regression
reg_df = inf_df.copy()
reg_df, targets_reg = the_pipeline.feature_engineering_reg(
	reg_df, inference=True
)
reg_df = reg_df.assign(
	timestamp=lambda x: pd.to_datetime(
		x[["year", "month", "day", "hour", "minute"]]
	)
)

# [!]
# Feature Selection -- Regression
to_drop = [
	*the_config.REGRESSION["to_drop"],
	"timestamp"
]
to_drop += [c for c in reg_df.columns if c.lower().startswith("conf_")]
X_reg = reg_df.drop(columns=to_drop)

logger.debug(f"[LIST] Targets ({len(targets_reg)}): {targets_reg}")

### Train-Test Split

In [ ]:
# [!]
# Train-Test Split -- Classification
X_test_clf = X_clf.iloc[-the_config.INF_ROWS:]

logger.debug(f"{'[LIST]':<8}{'[CLF]':<6}{'Features:':<10}{X_test_clf.columns.tolist()}")
logger.debug(f"{'[TEST]':<8}{'[CLF]':<6}{'Shape:':<10}{X_test_clf.shape}")

In [ ]:
# [!]
# Train-Test Split -- Regression
X_test_reg = X_reg.iloc[-the_config.INF_ROWS:]

logger.debug(f"{'[LIST]':<8}{'[REG]':<6}{'Features:':<10}{X_test_reg.columns.tolist()}")
logger.debug(f"{'[TEST]':<8}{'[REG]':<6}{'Shape:':<10}{X_test_reg.shape}")

### Models

In [ ]:
# [!]
# Model -- Classification
clf_blob = next(
    blob for blob in GCS_BUCKET.list_blobs(
        prefix=the_config.GCS_PREFIX_LATEST
    )
    if blob.name.endswith("_classifier.joblib")
)
clf = joblib.load(
    io.BytesIO(clf_blob.download_as_bytes())
)
display(clf)

# [!]
# Model -- Regression
reg_blob = next(
    blob for blob in GCS_BUCKET.list_blobs(
        prefix=the_config.GCS_PREFIX_LATEST
    )
    if blob.name.endswith("_regressor.joblib")
)
reg = joblib.load(
    io.BytesIO(reg_blob.download_as_bytes())
)
display(reg)

### Results

In [ ]:
the_pipeline.exec_inference(clf, reg, clf_df, reg_df)